# Repository edition: 06_channels

Run from this repository in your existing GPU environment. Source is in `../suites/06_channels/`. The launcher creates a versioned working copy. No embedded base64 code or installation is needed. Review configuration before launching; optional stages are not all enabled by default. See `../docs/EXPERIMENTS.md` for paper mapping.


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
SUPPLEMENTARY = next((p for p in _candidates if (p / "tools" / "notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    SUPPLEMENTARY = next((p / "supplementary" for p in _candidates if (p / "supplementary/tools/notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    raise FileNotFoundError("Open this notebook from inside the extracted supplementary repository.")
sys.path.insert(0, str(SUPPLEMENTARY / "tools"))
from notebook_sources import source_bundle as _source_bundle


# Content and selection channels: six experiments
Uses your existing environment and Hugging Face access. **No installation or login cells.**

The complete code is embedded in this notebook. Start fresh from PopQA, or optionally reuse an existing pattern-results directory or analysis ZIP. No prior results or activation caches are required.

Start with `split` and `geometry` on all imported evaluation prompts. All six experiments are implemented; schedule the other stages after inspecting the workload and initial results. This notebook does **not** assume the proposed “under an hour” timings.

The first-token margin is a derivative objective, not a factual-correctness criterion. Accepted-answer generations are scored separately. See the bundled README for mathematical definitions and limitations.


In [ ]:
from pathlib import Path
import base64, hashlib, json, os, subprocess, sys, zipfile, io, time

BUNDLE_SHA256 = hashlib.sha256(base64.b64decode(_source_bundle('06_channels'))).hexdigest()
BUNDLE = _source_bundle('06_channels')

payload = base64.b64decode(BUNDLE)
assert hashlib.sha256(payload).hexdigest() == BUNDLE_SHA256
CODE = Path.home() / ("threeways_channel_code_" + BUNDLE_SHA256[:12])
CODE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for item in archive.infolist():
        target = (CODE / item.filename).resolve()
        if CODE.resolve() not in target.parents:
            raise ValueError("Unsafe archive path")
    archive.extractall(CODE)
print("Code ready:", CODE)

def run_command(args):
    result = subprocess.run(args, cwd=CODE, capture_output=True, text=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)
    result.check_returncode()
    return result


## 1. Select fresh data or optional saved input
Leave `INPUT = None` to create a fresh PopQA evaluation cohort. Alternatively set a saved directory or ZIP. Nothing is installed; fresh loading uses the datasets package in your existing environment.


In [ ]:
INPUT = None  # Start fresh from PopQA; no previous results needed.
# Example after uploading your compact backup:
# INPUT = Path("/home/ubuntu/patterns_v1_analysis_20261102_120000.zip")

RESULTS = Path.home() / "threeways_channel_results" / "channels_fresh_v1"
RESULTS.mkdir(parents=True, exist_ok=True)
PROMPTS = RESULTS / "prompts.jsonl"
CONFIG_PATH = RESULTS / "configuration.json"
LOG = RESULTS / "execution.log"

# Reuse an existing working environment. No installs or authentication.
candidates = [Path("/home/ubuntu/venvs/threeways/bin/python"), Path(sys.executable)] + [
    Path.home() / name / "bin/python"
    for name in ["threeways_followup_env", "threeways_causal_env", "threeways_pattern_env"]
]
check = "import torch, transformers, numpy, pandas, matplotlib, huggingface_hub; assert transformers.__version__ == '4.48.3', 'Expected tested Transformers 4.48.3'; assert torch.cuda.is_available(), 'CUDA unavailable'; print(torch.cuda.get_device_name())"
PYTHON = None
errors = []
for candidate in dict.fromkeys(candidates):
    if not candidate.exists():
        continue
    result = subprocess.run([str(candidate), "-c", check], capture_output=True, text=True)
    if result.returncode == 0:
        PYTHON = candidate
        print("Using:", PYTHON, "| GPU:", result.stdout.strip())
        break
    errors.append(str(candidate) + ": " + result.stderr[-1200:])
if PYTHON is None:
    raise RuntimeError("No compatible existing environment found. Point candidates to your previous working Python.\n" + "\n".join(errors))
if INPUT is not None:
    assert INPUT.exists(), "Set INPUT to an existing path, or None to start fresh."


## 2. Select models, cohort, and stages
Fresh mode selects up to 100 evaluation facts per relation across six relations, with five templates per valid fact (up to 3,000 prompts). Subjects are separated from calibration examples, and no model outcomes are used to select facts. This is a new cohort and may differ from your previous run. Defaults cover all generated/imported evaluation prompts for the split/geometry stages. The expensive perturbation/editing stages use 32 facts selected without looking at outcomes; training uses 128 facts. These sample sizes are explicit, editable protocol choices.

Use `STAGES = ["split", "fisher", "geometry", "perturb", "frontier", "training"]` to request all six. You can add stages later with the same configuration and output folder. Changing configuration requires a new output folder.

Optional original regime labels: set `regimes_csv` to a CSV with `model,id,template,regime`. Without this, output groups are answer successes and unlabelled answer failures; the code does not invent causal-stage labels.


In [ ]:
CONFIG = json.loads((CODE / "config.json").read_text())
STAGES = ["split", "geometry"]

# Keep all four models, or select a subset BEFORE starting a new output folder.
# CONFIG["models"] = CONFIG["models"][:1]
# CONFIG["regimes_csv"] = "/home/ubuntu/original_regimes.csv"

# These do not limit the split/geometry/Fisher cohort:
# CONFIG["probe_facts"] = 32
# CONFIG["training_facts"] = 128
# CONFIG["max_prompts"] = None  # all available evaluation prompts

if CONFIG_PATH.exists() and json.loads(CONFIG_PATH.read_text()) != CONFIG:
    raise RuntimeError("Configuration differs from this run. Use a new RESULTS folder.")
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2))

# Import once. Worker fingerprints the selected cohort on first execution.
if not PROMPTS.exists():
    if INPUT is None:
        command = [str(PYTHON), "-m", "channel_study.fresh", "--out", str(RESULTS),
                   "--seed", str(CONFIG["seed"]), "--evaluation-per-relation", "100"]
    else:
        command = [str(PYTHON), "-c",
            "from channel_study.io import import_prompts; import sys; import_prompts(sys.argv[1],sys.argv[2])",
            str(INPUT), str(PROMPTS)]
    run_command(command)
rows = [json.loads(line) for line in PROMPTS.read_text().splitlines() if line.strip()]
from collections import Counter
print("Imported prompts:", len(rows), "| facts:", len({r['id'] for r in rows}))
print("Template coverage:", dict(Counter(r['template'] for r in rows)))

run_command([str(PYTHON), "-m", "channel_study.plan", "--config", str(CONFIG_PATH),
                "--prompts", str(PROMPTS), "--stages", *STAGES])


## 3. Launch in the background
The worker survives closing the browser or interrupting a monitor cell. The GPU instance must stay powered on. Output is written incrementally. Run only one launcher for this results directory.


In [ ]:
import fcntl
with (RESULTS / "run.lock").open("a+") as handle:
    try:
        fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
    except BlockingIOError:
        raise RuntimeError("A worker is already running for this directory. Use the monitor.")
# Also guard the short period before a newly launched worker takes its lock.
if "supervisor" in globals() and supervisor.poll() is None:
    raise RuntimeError("This notebook already launched a live supervisor. Use the monitor.")
if (RESULTS / "STOP").exists():
    raise RuntimeError("STOP exists. Use the resume cell after verifying the worker has stopped.")
(RESULTS / "exit.json").unlink(missing_ok=True)
command = [str(PYTHON), "-u", "-m", "channel_study.supervisor",
           "--out", str(RESULTS), "--config", str(CONFIG_PATH),
           "--prompts", str(PROMPTS), "--stages", *STAGES]
with (RESULTS / "supervisor.log").open("a") as logfile:
    supervisor = subprocess.Popen(command, cwd=CODE, stdout=logfile,
                                  stderr=subprocess.STDOUT, start_new_session=True)
print("Supervisor PID:", supervisor.pid)
print("Log:", LOG)


## 4. Monitor (rerun whenever you want)
This does not wait for the experiment to finish. An exit code of zero with `stopped: true` means a requested stop, not completion of the campaign.


In [ ]:
if LOG.exists():
    with LOG.open("rb") as f:
        f.seek(max(0, LOG.stat().st_size - 14000))
        print(f.read().decode(errors="replace"))
else:
    print("Waiting for worker log.")
if (RESULTS / "exit.json").exists():
    print("Worker exit:", json.loads((RESULTS / "exit.json").read_text()))


## 5. Build/inspect partial summaries
Optional while the worker runs. Only completed prompt stages enter these reports. Report generation consumes CPU but does not stop the GPU worker. All generations and Fisher draws remain in their original per-prompt JSON files.


In [ ]:
run_command([str(PYTHON), "-m", "channel_study.run", "--config", str(CONFIG_PATH),
                "--prompts", str(PROMPTS), "--out", str(RESULTS), "--stages", "report"])
import csv
for stage in STAGES:
    path = RESULTS / "reports" / (stage + "_summary.csv")
    if path.exists():
        print("\n", stage.upper(), "— first 8 summary rows")
        with path.open() as f:
            reader = csv.DictReader(f)
            for i, row in enumerate(reader):
                if i >= 8: break
                print(row)
print("Complete CSVs and PDF figures:", RESULTS / "reports")


## 6. Stop gracefully — run ONLY when you want to stop
The current operation finishes, then the worker stops at a safe boundary. Monitor until `exit.json` appears before powering down or resuming.


In [ ]:
# Uncomment only when you want to STOP:
# (RESULTS / "STOP").touch()
# print("Stop requested. Monitor for exit.json before closing the GPU instance.")


## 7. Resume or add another experiment
After the worker has exited, set `STAGES` above to the next stages and rerun the workload-plan and launcher cells. Keep `CONFIG`, `PROMPTS`, and `RESULTS` unchanged. To clear a previous stop request, uncomment and run the following cell first.


In [ ]:
# Uncomment only after the worker has stopped:
# with (RESULTS / "run.lock").open("a+") as handle:
#     fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
#     (RESULTS / "STOP").unlink(missing_ok=True)
# print("Ready to resume. Rerun the launcher with the desired STAGES.")


## 8. Save relevant results before terminating the instance
After stopping the worker, this archives this study's results and small origin vectors. No pretrained model weights or large activation caches are written by this study. Download the ZIP to your computer before terminating the instance. This does not back up Hugging Face model downloads or your Python environment.


In [ ]:
from datetime import datetime
BACKUP = RESULTS.parent / (RESULTS.name + "_results_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".zip")
with (RESULTS / "run.lock").open("a+") as handle:
    fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
    with zipfile.ZipFile(BACKUP, "w", zipfile.ZIP_DEFLATED, compresslevel=3) as archive:
        for folder, dirs, files in os.walk(RESULTS):
            dirs[:] = [d for d in dirs if d != "__pycache__"]
            for name in files:
                p = Path(folder) / name
                if p.suffix in {".json", ".jsonl", ".csv", ".pdf", ".py", ".txt", ".log", ".pt"}:
                    archive.write(p, p.relative_to(RESULTS))
with zipfile.ZipFile(BACKUP) as archive:
    assert archive.testzip() is None
print("Verified backup:", BACKUP, "| MiB:", round(BACKUP.stat().st_size / 2**20, 1))
from IPython.display import display, FileLink
display(FileLink(os.path.relpath(BACKUP, Path.cwd())))
